In [ ]:
# %pip install -q langchain-community
# %pip install -q pypdf
# %pip install -q langchain-huggingface
# %pip install -q chromadb
# %pip install -q faiss-cpu
# %pip install -q --upgrade opentelemetry-api opentelemetry-sdk

In [2]:
# Data Ingestion
from langchain_community.document_loaders import TextLoader
loader = TextLoader("sample.txt")
text_documents = loader.load()
text_documents

/tmp/ipykernel_20652/2776599453.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


[Document(metadata={'source': 'sample.txt'}, page_content='LanceDB\nA wrapper for an open-source database for vector-search with persistent storage. It simplifies retrieval, filtering, and management of embeddings.\nLangChain is a framework designed for building applications with large language models (LLMs) by chaining together various components. It supports a range of functionalities including memory, agents, and chat models, enabling developers to create context-aware applications.\nLangChain streamlines these stages (in figure above) by providing pre-built components and tools for integration, memory management, and deployment, allowing developers to focus on application logic rather than underlying complexities.')]

In [3]:
## Web based loader
from langchain_community.document_loaders import WebBaseLoader
import bs4

### load, chunk and index the content of the html page

loader = WebBaseLoader(
  web_paths = ("https://reference.langchain.com/python/langchain-community/document_loaders",),
  bs_kwargs = dict(parse_only = bs4.SoupStrainer(
    class_ = ("markdown-content markdown-compact mt-3 text-foreground-primary text-lg","text-xl font-heading font-semibold text-foreground mb-4")
  ))
)
text_documents = loader.load()

In [4]:
text_documents

[Document(metadata={'source': 'https://reference.langchain.com/python/langchain-community/document_loaders'}, page_content='Document Loaders  are classes to load Documents.\nDocument Loaders are usually used to load a lot of Documents in a single run.\nClass hierarchy:\n.. code-block::\nBaseLoader --> <name>Loader  # Examples: TextLoader, UnstructuredFileLoader\n\nMain helpers:\n.. code-block::\nDocument, <name>TextSplitter\nAttributesClassesModules')]

In [5]:
## Pdf reader
from langchain_community.document_loaders import PyPDFLoader
loader=PyPDFLoader('attention.pdf')
docs=loader.load()

In [6]:
docs

[Document(metadata={'producer': 'pdfcpu v0.12.1 dev', 'creator': 'PyPDF', 'creationdate': '2026-09-23T05:27:15+00:00', 'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin', 'book': 'Advances in Neural Information Processing Systems 30', 'created': '2017', 'date': '2017', 'description': 'Paper accepted and presented at the Neural Information Processing Systems Conference (http://nips.cc/)', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallel

In [7]:
## Chunking (text splitting)
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200) #10,20% overlap
documents = text_splitter.split_documents(docs) #attention.pdf document is split recursively
documents[:5]

[Document(metadata={'producer': 'pdfcpu v0.12.1 dev', 'creator': 'PyPDF', 'creationdate': '2026-09-23T05:27:15+00:00', 'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin', 'book': 'Advances in Neural Information Processing Systems 30', 'created': '2017', 'date': '2017', 'description': 'Paper accepted and presented at the Neural Information Processing Systems Conference (http://nips.cc/)', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallel

In [8]:
## Vector embedding and vector store
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
#from langchain_openai import OpenAIEmbeddings

#db = Chroma.from_documents(documents,OpenAIEmbeddings())
embedding_model = HuggingFaceEmbeddings(
  model = "sentence-transformers/all-MiniLM-L6-v2"
)
db = Chroma.from_documents(
  documents,
  embedding_model
)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [9]:
query = "Who are the authors of attention is all you need?"
retrieved_results = db.similarity_search(query)
print(retrieved_results[0].page_content)

2017.
[16] Yoon Kim, Carl Denton, Luong Hoang, and Alexander M. Rush. Structured attention networks.
In International Conference on Learning Representations , 2017.
[17] Diederik Kingma and Jimmy Ba. Adam: A method for stochastic optimization. In ICLR, 2015.
[18] Oleksii Kuchaiev and Boris Ginsburg. Factorization tricks for LSTM networks. arXiv preprint
arXiv:1703.10722, 2017.
[19] Zhouhan Lin, Minwei Feng, Cicero Nogueira dos Santos, Mo Yu, Bing Xiang, Bowen
Zhou, and Yoshua Bengio. A structured self-attentive sentence embedding. arXiv preprint
arXiv:1703.03130, 2017.
[20] Samy Bengio Łukasz Kaiser. Can active memory replace attention? In Advances in Neural
Information Processing Systems, (NIPS), 2016.
10
amethystvc07@gmail.com


In [10]:
# FAISS Vector store
from langchain_community.vectorstores import FAISS
db = FAISS.from_documents(
  documents = documents[:15],
  embedding = embedding_model
)

## Retriever Chain

In [ ]:
# Setup ollama on collab
# !curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# !ollama --version

In [ ]:
# ## Start the Ollama server
# import subprocess
# import time

# ollama_process = subprocess.Popen(
#     ["ollama", "serve"],
#     stdout=subprocess.DEVNULL,
#     stderr=subprocess.DEVNULL
# )

# time.sleep(5)
# print("Ollama server started.")

Ollama server started.


In [ ]:
# !ollama pull llama2

In [ ]:
# ## verify ollama is working
# !ollama list

NAME             ID              SIZE      MODIFIED               
llama2:latest    78e26419b446    3.8 GB    Less than a second ago    


In [16]:
from langchain_community.llms import Ollama
## Load Ollama LAMA2 LLM model
llm=Ollama(model="llama2")
llm

/tmp/ipykernel_20652/2277366689.py:3: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm=Ollama(model="llama2")


Ollama(metadata={'lc_versions': {'langchain-core': '1.6.12', 'langchain': '1.4.6'}})

In [17]:
## Design ChatPrompt Template
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_template("""
Answer the following question based only on the provided context.
Think step by step before providing a detailed answer.
I will tip you $1000 if the user finds the answer helpful.
<context>
{context}
</context>
Question: {input}""")

In [18]:
"""
Retrievers: A retriever is an interface that returns documents given
 an unstructured query. It is more general than a vector store.
 A retriever does not need to be able to store documents, only to
 return (or retrieve) them. Vector stores can be used as the backbone
 of a retriever, but there are other types of retrievers as well.
 https://python.langchain.com/docs/modules/data_connection/retrievers/
"""

## Create the retriever
retriever=db.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x78f322d1cec0>, search_kwargs={})

In [19]:
"""
Retrieval chain:This chain takes in a user inquiry, which is then
passed to the retriever to fetch relevant documents. Those documents
(and original inputs) are then passed to an LLM to generate a response
https://python.langchain.com/docs/modules/chains/
"""
# Build the retrieval pipeline (REPLACES the old chain helpers)
from langchain_core.runnables import RunnablePassthrough

retrieval_chain = (
    {
        "context": retriever | (
            lambda docs: "\n\n".join(
                doc.page_content for doc in docs
            )
        ),
        "input": RunnablePassthrough()
    }
    | prompt
    | llm
)

In [20]:
# Ask a question (SAME question as your original code)
response = retrieval_chain.invoke("Scaled Dot-Product Attention")

In [23]:
print(response)

Based on the provided context, the answer to the question "Scaled Dot-Product Attention" is:

Scaled Dot-Product Attention is a type of attention mechanism used in transformer models, particularly in the Transformer architecture. It is called "scaled dot-product attention" because it computes the compatibility function using a dot product attention mechanism, but with a scaling factor of 1√dk.

The input to the scaled dot-product attention mechanism consists of queries, keys, and values, all of which are vectors of dimension dk. The scaling factor of 1√dk is applied to the dot products of the queries and keys before computing the attention weights. This allows the model to attend to a larger range of positions in the input sequence without the risk of the attention weights becoming too large and causing numerical instability.

The output of the scaled dot-product attention mechanism is a vector of dimension dv, where each element represents the attention weight for a particular positio